In [115]:
!pip install krippendorff

In [116]:
import pandas as pd
import os
import glob
import krippendorff
from sklearn.metrics import cohen_kappa_score

In [117]:
DATA_DIR = '../../data/v3/annotations/disambiguation/'

# Functions

In [140]:
def compute_krippendorff_alpha(df, col1, col2):
    cats = pd.Categorical(pd.concat([df[col1], df[col2]]).dropna())
    mapping = {cat: i for i, cat in enumerate(cats.categories)}

    coded_0 = df[col1].map(mapping).astype("float")
    coded_1 = df[col2].map(mapping).astype("float")
    mask = coded_0.notna() & coded_1.notna()

    matrix = pd.DataFrame({
        "col_0": coded_0[mask],
        "col_1": coded_1[mask]
    }).T.values

    alpha = krippendorff.alpha(
        reliability_data=matrix,
        level_of_measurement='nominal'
    ) 
    return alpha

def compute_cohen_kappa(df, col1, col2):
    return cohen_kappa_score(df[col1].values, df[col2].values)

# First round

In [141]:
data_annotator = {}
for id, fn in enumerate(glob.glob(os.path.join(DATA_DIR, '*.xlsx'))):
    if 'final' in fn:
        continue
    print(id, fn)
    obj = pd.read_excel(fn, sheet_name=None, usecols=['name', 'title1', 'title2', 'manual_check'])
    key = 'Sheet1' if 'Sheet1' in obj else 'name_test'
    data_annotator[id] = obj[key].set_index('name')


0 ../../data/v3/annotations/disambiguation/name_test_Mingrong.xlsx
1 ../../data/v3/annotations/disambiguation/Name_test_Liu.xlsx


In [142]:
[df.shape for key, df in data_annotator.items()]

[(80, 3), (80, 3)]

In [143]:
col = ['manual_check']
df_annotations = data_annotator[0][col].rename(columns={col[0]:'col_0'})
for key, df in data_annotator.items():
    if key == 0:
        continue
    print(key)
    df_annotations = df_annotations.join(df[col].rename(columns={col[0]:f'col_{key}'}))

1


In [144]:
df_annotations.loc[:, 'agreement'] = df_annotations.apply(lambda row: row.col_0 == row.col_1, axis=1)
df_annotations.sample(10)

,col_0,col_1,agreement
name,,,
陈海峰,not_match,not_match,True
辛晶,match,match,True
魏强,not_match,not_match,True
杨利霞,match,match,True
韩波,match,match,True
丁屹,match,match,True
常相辉,match,match,True
杨雪滢,not_match,not_match,True
黄庆举,not_match,match,False


In [145]:
df_annotations.query("agreement==False")

,col_0,col_1,agreement
name,,,
崔健,not_match,NaN,False
朱爱民,match,NaN,False
李军,not_match,NaN,False
杨洪,not_match,NaN,False
王强华,match,not_match,False
贺腾,not_match,match,False
郝三如,match,NaN,False
陈文龙,not_match,NaN,False
陈绍英,match,not_match,False


In [146]:
df_annotations.agreement.value_counts()

agreement
True     70
False    10
Name: count, dtype: int64

In [147]:
df_annotations.fillna('-', inplace=True)

## Krippendorff


In [148]:
compute_krippendorff_alpha(df_annotations, 'col_0', 'col_1')

0.755083179297597

## Cohen Kappa

In [149]:
compute_cohen_kappa(df_annotations, 'col_0', 'col_1')

0.754601226993865

# Second Round

In [150]:
fn = os.path.join(DATA_DIR, 'Name_test_Liu_Mingrong_final.xlsx')
df_final = pd.read_excel(fn, sheet_name=None, usecols=['name', 'title1', 'title2', 'manual_check_Mingrong', 'manual_check_liu', 'Final'])['Sheet1']

data_final = df_final.set_index('name')[['Final']].rename(columns={'Final':'col_0'})
data_final = data_final.join(data_final, rsuffix='1').rename(columns={'col_01':'col_1'})
data_final.head(5)

,col_0,col_1
name,,
丁屹,match,match
丁菊仁,match,match
万亚,match,match
万贤纲,match,match
万钧,not_match,not_match


In [151]:
compute_krippendorff_alpha(data_final, 'col_0', 'col_1')

1.0

In [152]:
compute_cohen_kappa(data_final, 'col_0', 'col_1')

1.0